In [5]:
import torch
import torch.nn as nn
import joblib
import numpy as np
import pandas as pd


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Define the same multilayer perceptron architecture as used during training
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(401, 1800),
            nn.ReLU(),
            nn.Dropout(0.046),
            nn.Linear(1800, 1800),
            nn.ReLU(),
            nn.Dropout(0.046),
            nn.Linear(1800, 5)
        )

    def forward(self, x):
        return self.model(x)


# Load the fitted input and output standardization objects
x_scaler = joblib.load("x_scaler.pkl")
y_scaler = joblib.load("y_scaler.pkl")


# Load the trained model parameters
model = MLP().to(device)
state = torch.load("trained_model.pth", map_location=device)
model.load_state_dict(state)
model.eval()


# Load new input data for prediction
new_data = pd.read_csv("Prediction_input.csv")
X_new = new_data.values


# Check whether the input feature dimension matches the training configuration
if X_new.shape[1] != 401:
    raise ValueError(
        f"Input feature dimension should be 401, but the current input has {X_new.shape[1]} columns. "
        "Please verify that the number and order of features are consistent with those used during training."
    )


# Apply the same standardization procedure used in the training stage
X_new_scaled = x_scaler.transform(X_new)
X_new_tensor = torch.tensor(X_new_scaled, dtype=torch.float32).to(device)


# Perform forward prediction without gradient calculation
with torch.no_grad():
    Y_pred_scaled = model(X_new_tensor).cpu().numpy()


# Transform the predicted outputs back to the original scale
Y_pred = y_scaler.inverse_transform(Y_pred_scaled)
Y_pred = np.abs(Y_pred)


# Save the prediction results
pred_df = pd.DataFrame(Y_pred, columns=["Rpop", "Rdep", "NT", "Rx", "Reh"])
pred_df.to_csv("prediction_result.csv", index=False)

print("Prediction completed. The results were saved to prediction_result.csv.")

Prediction completed. The results were saved to prediction_result.csv.


C:\Users\Scarlett\AppData\Local\Temp\ipykernel_164\1342764639.py:36: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load("trained_model.pth", map_location=devic